Link:
https://www.kaggle.com/datasets/laotse/credit-risk-dataset

# Description

Detailed data description of Credit Risk dataset:

| **Feature Name**           | **Categorical values** | **Description**                                                                           |
|----------------------------|------------------------|-------------------------------------------------------------------------------------------|
| person_age                 |                        | Age of the individual applying for the loan                                               |
| person_income              |                        | Annual income of the individual                                                           |
| person_home_ownership      |                        | Type of home ownership of the individual                                                  |
|                            | rent                   | The individual is currently renting a property                                            |
|                            | mortgage               | The individual has a mortgage on the property they own                                    |
|                            | own                    | The individual owns their home outright                                                   |
|                            | other                  | Other categories of home ownership that may be specific to the dataset                    |
| person_emp_length          |                        | Employment length of the individual in years                                              |
| loan_intent                |                        | The intent behind the loan application                                                    |
| loan_grade                 |                        | The grade assigned to the loan based on the creditworthiness of the borrower              |
|                            | A                      | The borrower has a high creditworthiness, indicating low risk                             |
|                            | B                      | The borrower is relatively low-risk, but not as creditworthy as Grade A                   |
|                            | C                      | The borrower's creditworthiness is moderate                                               |
|                            | D                      | The borrower is considered to have higher risk compared to previous grades                |
|                            | E                      | The borrower's creditworthiness is lower, indicating a higher risk                        |
|                            | F                      | The borrower poses a significant credit risk                                              |
|                            | G                      | The borrower's creditworthiness is the lowest, signifying the highest risk                |
| loan_amnt                  |                        | The loan amount requested by the individual                                               |
| loan_int_rate              |                        | The interest rate associated with the loan                                                |
| loan_status                |                        | Loan status, where 0 indicates non-default and 1 indicates default                        |
| loan_percent_income        |                        | The percentage of income represented by the loan amount                                   |
| cb_person_default_on_file  |                        | The individual has a history of defaults on their credit file. Y -has, N - does not have. |
| cb_preson_cred_hist_length |                        | The length of credit history for the individual                                           |

# Set constants

In [ ]:
RANDOM_SEED = 64

# Download dataset

In [ ]:
#import kagglehub
# Download latest version
#path = kagglehub.dataset_download("laotse/credit-risk-dataset")
#print("Path to dataset files:", path)

In [ ]:
%matplotlib inline

In [ ]:
import pandas as pd
credit_data = pd.read_csv('../data/credit_risk_dataset.csv')

# EDA

In [ ]:
credit_data.head(10)

In [ ]:
credit_data.describe().T

In [ ]:
credit_data.info()

In [ ]:
#credit_data.corr()

In [ ]:
#heavy-weight automatic EDA
from ydata_profiling import ProfileReport
#ProfileReport(credit_data) # uncommnet to see html report on dataset

## Mising data

person_emp_length - Employment length (in years) - 895 - 2.7%

loan_int_rate - Interest rate - 3116 - 9.6%



## Data with suspicious values

person_age - Age of the individual applying for the loan - Maximum = 144

person_emp_length - Employment length of the individual in years - Maximum = 123 

## Deal with missing data
Assume that if there is no information about person employment it means he did not work.
Assume that if there is no information about interest rate then it can be mean by value.

In [ ]:
fixed_credit_data = credit_data.copy(deep=True)
fixed_credit_data['person_emp_length'] = fixed_credit_data['person_emp_length'].fillna(fixed_credit_data['person_emp_length'].median())

In [ ]:
fixed_credit_data['loan_int_rate'] = fixed_credit_data['loan_int_rate'].fillna(fixed_credit_data['loan_int_rate'].mean())

## Remove duplicate rows
Assume duplicates are input error.

In [ ]:
fixed_credit_data = fixed_credit_data.drop_duplicates()

## Deal with suspicious data 

In [ ]:
MAXIMUM_OF_AGE = 90
AGE_OF_WORK_STARTED = 14
MAXIMUM_OF_EMPLOYEMENT_YEARS = MAXIMUM_OF_AGE - AGE_OF_WORK_STARTED
if 'person_age' in fixed_credit_data.columns:
    fixed_credit_data = fixed_credit_data[fixed_credit_data['person_age'] <= MAXIMUM_OF_AGE] 
if 'person_emp_length' in fixed_credit_data.columns:
    fixed_credit_data = fixed_credit_data[fixed_credit_data['person_emp_length'] <= MAXIMUM_OF_EMPLOYEMENT_YEARS]

## Deal with multicorelation


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 2. Compute the correlation matrix
corr_matrix = fixed_credit_data.corr(numeric_only=True)

# 3. Plot the heatmap
plt.figure(figsize=(10, 8)) # Set the figure size
sns.heatmap(
    corr_matrix,
    annot=True,         # Annotate cells with the correlation values
    cmap='coolwarm',    # Choose a color map (e.g., 'coolwarm', 'viridis', 'Blues')
    fmt=".2f",          # Format the annotations to two decimal places
    vmin=-1,            # Set the minimum value for the color bar
    vmax=1,             # Set the maximum value for the color bar
    center=0,           # Center the color bar at 0
    square=True,        # Ensure cells are square
    linewidths=.5,      # Add lines between cells
    cbar_kws={"shrink": .5} # Shrink the color bar height
)

plt.title('Fixed dataframe Correlation Matrix Heatmap')
plt.show()

Assume that high correlation between age and person's credit history length means that age holds not much useful information and we can keep only person's credit history length.

In [ ]:
#feature engineering
fixed_credit_data["cred_hist_to_age"] = fixed_credit_data['cb_person_cred_hist_length']/fixed_credit_data['person_age']
#fixed_credit_data["employment_to_age"] = fixed_credit_data['person_emp_length']/fixed_credit_data['person_age']
#fixed_credit_data["years_unemployed"] = fixed_credit_data['person_age'] - fixed_credit_data['person_emp_length']
fixed_credit_data = fixed_credit_data.drop('cb_person_cred_hist_length', axis=1)
fixed_credit_data = fixed_credit_data.drop('person_age', axis=1)

Assume that loan percent income is irrelevant to it's default.

But it can be useful for other metric - money loss. 
Assume that in case of default that was not anticipated we lose loan body plus it's interest and in case of non-default client that was marked as to be default we lose only interest that we could gain. We also assume that all loans are given for the year because we do not have time frame for loans.

In [ ]:
fixed_credit_data["possible_loss"] = fixed_credit_data.apply(
    lambda x: x['loan_amnt']*(1 + x['loan_int_rate']/100) if x['loan_status'] == 1 
    else x['loan_amnt']*x['loan_int_rate']/100, axis=1
)
#fixed_credit_data = fixed_credit_data.drop('loan_percent_income', axis=1)

Let us check value of 'loan_percent_income' columns, to understand what it means.

In [ ]:
#fixed_credit_data['check_loan_percent_income'] = round(fixed_credit_data['loan_amnt']/fixed_credit_data['person_income'],2)
#fixed_credit_data['check_percent_income_dif'] = fixed_credit_data['check_loan_percent_income']/fixed_credit_data['loan_percent_income']

In [ ]:
#fixed_credit_data[fixed_credit_data['check_loan_percent_income'] != fixed_credit_data['loan_percent_income']].head(20)

In [ ]:
#print(1005/32581*100)

Take note, around 1005 rows has a value for `loan_percent_income` column that is only close, but is not equal to `['loan_amnt']/['person_income']`. I do not have an explanation for this. For a time being this problem will be ignored.

In [ ]:
from ydata_profiling import ProfileReport
fixed_credit_data_without_loss = fixed_credit_data.loc[ : , fixed_credit_data.columns != 'possible_loss']
#ProfileReport(fixed_credit_data_without_loss) # uncomment to see report of fixed dataset

# Deal with categorical data

In [ ]:
fixed_credit_data_onehot_encoding  = pd.get_dummies(fixed_credit_data, drop_first=True)

# Make train and test parts

In [ ]:
from sklearn.model_selection import train_test_split

y = fixed_credit_data_onehot_encoding["loan_status"]
X = fixed_credit_data_onehot_encoding.drop(columns=["loan_status"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
X_train_loss = X_train['possible_loss']
X_test_loss  =  X_test['possible_loss']
X_train = X_train.drop('possible_loss', axis=1)
X_test  =  X_test.drop('possible_loss', axis=1)

In [ ]:
print(type(X_train))

In [ ]:
X_train.head()

In [ ]:
X_train.isna().sum()

## Normalise data values

Link:
https://www.kaggle.com/datasets/laotse/credit-risk-dataset

In [ ]:
# TODO: We can define our own preprocessing class
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
numerical_values = ['person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'cb_person_cred_hist_length']
categorical_values = ['person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT', 
                      'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 
                      'loan_intent_PERSONAL', 'loan_intent_VENTURE	loan_grade_B', 'loan_grade_C', 'loan_grade_D', 
                      'loan_grade_E', 'loan_grade_F', 'loan_grade_G', 'cb_person_default_on_file_Y'
                      ]
#X_train_scaled = scaler.fit_transform(X_train[numerical_values])
#X_test_scaled = scaler.transform(X_test[numerical_values])
#X_train_scaled[categorical_values] = X_train[categorical_values]
#X_test_scaled[categorical_values] = X_test[categorical_values]

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
print(X_train_scaled[2])

In [ ]:
from sklearn.metrics import root_mean_squared_error, precision_score, accuracy_score, recall_score, f1_score, confusion_matrix

def train_classifier(clf, X_train, y_train, X_test, y_test):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Calculate confusion matrix to get TN and FP for specificity
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp)
    
    return accuracy, precision, recall, specificity, f1, y_pred

In [ ]:
def calculate_possible_loss(y_true, y_pred, possible_loss:pd.Series) -> float:
    sum_awaited_loss = 0
    data = {'y_pred': y_pred, 'y_true': y_true}
    df = pd.DataFrame(data)
    df.insert(0, "possible_loss", possible_loss, allow_duplicates=False)
    sum_awaited_loss = df[df["y_pred"] != df["y_true"]]["possible_loss"].sum()
    return sum_awaited_loss

In [ ]:
def custom_money_loss_scorer(y_true, y_pred):
    return -1*calculate_possible_loss(y_true, y_pred, possible_loss=X_train_loss)

# Logistic regression on k-fold, roc-auc metric + money loss metric

In [ ]:
import numpy as np
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression

log = LogisticRegression(solver = 'liblinear', random_state = RANDOM_SEED)

log_roc_cv_scores = cross_val_score(
    log,
    X_train_scaled, y_train,
    cv=5, scoring="roc_auc"
)

print ("K-fold results for ROC-AUC:")
print ("\n".join(f"ROC AUC = {s:.4f}" for s in log_roc_cv_scores))
print (f"Mean ROC AUC on folds = {np.mean(log_roc_cv_scores) :.4f}")

money_loss_scorer = make_scorer(score_func=custom_money_loss_scorer, greater_is_better=True)

log_money_cv_scores = cross_val_score(
    log,
    X_train_scaled, y_train,
    cv=5, scoring=money_loss_scorer
)

print ("K-fold results for money loss:")
print ("\n".join(f"Expected money loss = {-1*s:.2f}" for s in log_money_cv_scores))
print (f"Mean Expected money loss on folds = {-1*np.mean(log_money_cv_scores) :.2f}")

In [ ]:
# Check our model on test data
accuracy, precision, recall, specificity, f1, y_pred = train_classifier(log, X_train_scaled, y_train, X_test_scaled, y_test)
probabilities = log.predict_proba(X_test_scaled)
probs_and_predictions = pd.DataFrame()
probs_and_predictions['prob_class0']=pd.Series(probabilities[:,0])
probs_and_predictions['prob_class1']=pd.Series(probabilities[:,1])
probs_and_predictions['pred_class']=pd.Series(y_pred)
print(probs_and_predictions[abs(probs_and_predictions['prob_class0'] - 0.5) <= 0.1].count())
money_loss = calculate_possible_loss(y_test, y_pred, X_test_loss)
print(f"Model Accuracy Score:    {accuracy:.4f}")
print(f"Model Precision Score:   {precision:.4f}")
print(f"Model Recall Score:      {recall:.4f}")
print(f"Model Specificity Score: {specificity:.4f}")
print(f"Model f1 Score: {f1:.4f}")
print(f"Awaited loss upon using model: {money_loss:.2f}")

# Try different ml models

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb

In [ ]:
svc = SVC(probability=True)
knc = KNeighborsClassifier() #algorithm='ball_tree', leaf_size=10, n_neighbors=18, p=1, weights='distance'
mnb = MultinomialNB()
dtc = DecisionTreeClassifier()
lrc = LogisticRegression()
rfc = RandomForestClassifier()
abc = AdaBoostClassifier()
bc = BaggingClassifier()
etc = ExtraTreesClassifier()
gbdt = GradientBoostingClassifier()
xgb = XGBClassifier()
cat = CatBoostClassifier(verbose=0) 
lgb = lgb.LGBMClassifier()

In [ ]:
# more long list of models
clfs = {
    'SVC': svc,
    'KN': knc, 
    #'NB': mnb, 
    'DT': dtc, 
    'LR': lrc, 
    'RF': rfc, 
    'AdaBoost': abc, 
    'BgC': bc, 
    'ETC': etc,
    'GBDT': gbdt,
    'xgb': xgb,
    'cat': cat,
    'lgb': lgb
}

In [ ]:
accuracy_scores = []
precision_scores = []
recall_scores = []
specificity_scores = []
f1_scores = []
money_lost = []

for name, clf in clfs.items():
    
    current_accuracy, current_precision, current_recall, current_specificity, current_f1, y_pred = train_classifier(
        clf, 
        X_train_scaled, 
        y_train, 
        X_test_scaled, 
        y_test
    )
    money_loss = calculate_possible_loss(y_test, y_pred, X_test_loss)

    predict_prob_method = getattr(clf, "predict_proba", None)
    if predict_prob_method is not None and callable(predict_prob_method):
        probabilities = clf.predict_proba(X_test_scaled)
        probs_and_predictions = pd.DataFrame()
        probs_and_predictions['prob_class0']=pd.Series(probabilities[:,0])
        probs_and_predictions['prob_class1']=pd.Series(probabilities[:,1])
        probs_and_predictions['pred_class']=pd.Series(y_pred)
        print(probs_and_predictions[abs(probs_and_predictions['prob_class0'] - 0.5) <= 0.1].count())

    print("For ",name)
    print("Accuracy - ",    current_accuracy)
    print("Precision - ",   current_precision)
    print("Recall - ",      current_recall)
    print("Specificity - ", current_specificity)
    print("F1 - ",          current_f1)
    print("Money loss - ",  money_loss)
    print()
    
    accuracy_scores.append(current_accuracy)
    precision_scores.append(current_precision)
    recall_scores.append(current_recall)
    specificity_scores.append(current_specificity)
    f1_scores.append(current_f1)
    money_lost.append(money_loss)

# Ensemble of best models 

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

cat = CatBoostClassifier(verbose=0)
lgb = LGBMClassifier()
xgb = XGBClassifier()
dtc = DecisionTreeClassifier()

ensemble = VotingClassifier(estimators=[('cat', cat), 
                                        ('lgb', lgb), 
                                        ('xgb', xgb), 
                                        ('dtc', dtc)], 
                            voting='soft', verbose=0,
                            flatten_transform=False  # Crucial for some conversion issues
                            )
current_accuracy, current_precision, current_recall, current_specificity, current_f1, y_pred  = train_classifier(ensemble, 
                                                                                           X_train_scaled, y_train, 
                                                                                           X_test_scaled, y_test)
money_loss = calculate_possible_loss(y_test, y_pred, X_test_loss)
print("For ensemble")

probabilities = ensemble.predict_proba(X_test_scaled)
probs_and_predictions = pd.DataFrame()
probs_and_predictions['prob_class0']=pd.Series(probabilities[:,0])
probs_and_predictions['prob_class1']=pd.Series(probabilities[:,1])
probs_and_predictions['pred_class']=pd.Series(y_pred)
print(probs_and_predictions[abs(probs_and_predictions['prob_class0'] - 0.5) <= 0.1].count())

print("Accuracy - ",    current_accuracy)
print("Precision - ",   current_precision)
print("Recall - ",      current_recall)
print("Specificity - ", current_specificity)
print("F1 - ",          current_f1)
print("Money loss - ",  money_loss)
print()

# Auto ML using AutoGluon

In [ ]:
!pip install autogluon

In [ ]:
AUTOGLUON_USES_MY_PREPROCESSING = False
from sklearn.model_selection import train_test_split

if not AUTOGLUON_USES_MY_PREPROCESSING:
    #split dataframe into train and test sets:
    train, test = train_test_split(credit_data, test_size=0.2, random_state=RANDOM_SEED) # we do not use our preprocessing, provide AutoGluon "raw" data
else: # OR use existing preprocessed data:
    train, test = pd.DataFrame(X_train_scaled, columns=X_train.columns), pd.DataFrame(X_test_scaled, columns=X_test.columns) , 
    train['loan_status'] = y_train.tolist()
    test['loan_status'] = y_test.tolist()

#print size of each set
print(train.shape, test.shape)

In [ ]:
from autogluon.tabular import TabularPredictor

#predictor = TabularPredictor(label='loan_status').fit(train_data=train, presets='best_quality_v150')

In [ ]:
# View the metadata of the processed features
#print(predictor.feature_metadata)

# Specifically view the types of features used
#print(predictor.feature_metadata.get_type_map_raw())

In [ ]:
#predictions = predictor.predict(test, transform_features=(not AUTOGLUON_USES_MY_PREPROCESSING))

In [ ]:
#money_loss = calculate_possible_loss(y_test, predictions, X_test_loss)

In [ ]:
#metrics = predictor.evaluate(test, silent=True, auxiliary_metrics=True, display=True, detailed_report=True)
#print("For AutoGluon " + ("with our preprocessing" if AUTOGLUON_USES_MY_PREPROCESSING else "with auto preprocessing"))
#print("Accuracy - ",    metrics['accuracy'])
#print("Precision - ",   metrics['precision'])
#print("Recall - ",      metrics['recall'])
#print("Specificity - ", metrics['confusion_matrix'][0][0]/(metrics['confusion_matrix'][0][0]+metrics['confusion_matrix'][0][1]))
#print("F1 - ",          metrics['f1'])
#print("Money loss - ",  money_loss)
#print()

In [ ]:
#predictor.leaderboard(test, silent=True)

# Export selected solution for production

In [ ]:
#recreate ensemble as clean model object
ensemble = VotingClassifier(estimators=[('cat', cat), 
                                        ('lgb', lgb), 
                                        ('xgb', xgb), 
                                        ('dtc', dtc)], 
                            voting='soft', verbose=0,
                            flatten_transform=False  # Crucial for some conversion issues
                        )

In [ ]:
from sklearn.pipeline import Pipeline
from my_processor import MyDataPreprocessor # it summarise best (in my humble opinion) processing 

pipe = Pipeline(
    [
        ('preprocessor', MyDataPreprocessor(MAXIMUM_OF_AGE, MAXIMUM_OF_EMPLOYEMENT_YEARS, AGE_OF_WORK_STARTED)), 
        ("ensemble", ensemble)
    ]
)
pipe.fit(credit_data.drop(columns=['loan_status']), y) # we will use for production ready model all available examples as train set

In [ ]:
import joblib

path_to_save_model = "../../development/model/credit_default_pipeline.joblib"

paths = joblib.dump(
    pipe,
    path_to_save_model,
    compress=True
)
print("Saved pipeine files to" + paths[0])